In [ ]:
!git clone https://github.com/stacks/stacks-project

In [ ]:
import glob
import os
import json
import re
from collections import defaultdict
from tqdm import tqdm
import numpy as np

In [ ]:
files_ = glob.glob('./stacks-project/*.tex')
files = []
for f in files_:
    if 'coding.tex' in f:
        continue
    files.append(f)

stems = [os.path.basename(f).split('.tex')[0] for f in files]

len(files)

In [ ]:
all_types = set()
for f in files:
    tex = open(f).read()
    stem = os.path.basename(f).split('.tex')[0]

    labels_ = re.findall(r'\label{([a-z|A-Z|0-9|\-]+)}', tex)
    for l in labels_:
        all_types.add(l.split('-')[0])

all_types

In [ ]:
def extract_refs(s):
    refs = re.findall(r'\ref{([^}]*)}', s)
    refs = [ref for ref in refs if all([t not in exclude_kinds for t in ref.split('-')])]
    for i in range(len(refs)):
        add_stem = True
        for stem_ in stems:
            if refs[i].startswith(stem_):
                add_stem = False
        if add_stem:
            refs[i] = '%s-%s' % (stem, refs[i])
    return refs

In [ ]:
def parse_proof(statement):
    contents = statement.strip().split('\n')
    contents = list(filter(lambda s: s != '', contents))
    refs = extract_refs(proof)

    return {
        'contents': contents,
        'refs': refs,
    }

In [ ]:
def parse_item(statement):
    lines = statement.strip().split('\n')
    start = 0
    label = None
    for i, line in enumerate(lines):
        if '\label' in line:
            label = re.findall(r'\label{([^}]*)}', line)[0]
            start = i+1
            break
    if label is None:
        raise ValueError('no label')
    label = '%s-%s' % (stem, label)
    contents = lines[start:]
    contents = list(filter(lambda s: s != '', contents))
    refs = extract_refs(statement)

    return {
        'label': label,
        'categories': [stem],
        'title': label,
        'contents': contents,
        'refs': refs,
    }

In [ ]:
theorem_kinds = ['theorem', 'lemma', 'proposition']
definition_kinds = ['definition']
other_kinds = ['remark', 'remarks']
all_ref_kinds = theorem_kinds + definition_kinds + other_kinds
exclude_kinds = [t for t in all_types if t not in all_ref_kinds]

kind2type = {}
for kind in theorem_kinds:
    kind2type[kind] = 'theorem'
for kind in definition_kinds:
    kind2type[kind] = 'definition'
for kind in other_kinds:
    kind2type[kind] = 'other'

theorems = []
definitions = []
others = []
label2id = {}
texs = []
cnt = 0

In [ ]:
for f in files:
    tex = open(f).read()
    texs.append(tex)
    stem = os.path.basename(f).split('.tex')[0]

    for kind in all_ref_kinds:
        splits = tex.split('\begin{%s}' % kind)[1:]
        for split in splits:
            item = {
                'id': cnt,
                'type': kind2type[kind],
            }
            cnt += 1

            statement, other = split.split('\end{%s}' % kind)
            item.update(parse_item(statement))

            if kind in theorem_kinds:
                proof = other.split('\end{proof}')[0]
                proof = re.findall(r'\begin{proof}(.*)', proof, re.DOTALL)
                assert len(proof) == 1
                proof = proof[0]
                proof = parse_proof(proof)
                item['proofs'] = [proof]

                theorems.append(item)

            elif kind in definition_kinds:
                definitions.append(item)

            elif kind in other_kinds:
                others.append(item)

            label2id[item['label']] = item['id']